# Task 11 — Rank Resumes Against Job Descriptions

**Objective:** score resumes against job descriptions, rank candidates, evaluate ranking quality against human judgments, and inspect five misranked examples.

This notebook builds a transparent baseline. Human suitability labels are used only after scoring for evaluation; they are not model inputs or ranking features.

## 1. Imports and project paths

Place the three input CSV files together in `data/task11/`, `data/`, or the project root. The notebook checks these locations. Outputs are saved under `phase-3/reports/`.

In [1]:
from pathlib import Path
import re, json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

ROOT = Path("/workspace")
if not (ROOT / "data").exists():
    raise FileNotFoundError(f"Expected Docker-mounted project root at {ROOT}; data/ was not found.")

required_files = ["job_descriptions.csv", "resumes.csv", "resume_job_labels.csv"]
possible_data_dirs = [ROOT / "data" / "task11", ROOT / "data", ROOT]
DATA_DIR = next((p for p in possible_data_dirs if all((p / f).exists() for f in required_files)), None)
if DATA_DIR is None:
    raise FileNotFoundError("Place job_descriptions.csv, resumes.csv and resume_job_labels.csv in data/task11/, data/, or the project root.")
REPORT_DIR = ROOT / "phase-3" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("Input directory:", DATA_DIR)

Project root: /workspace
Input directory: /workspace/data


## 2. Load and validate the three datasets

In [2]:
job_df = pd.read_csv(DATA_DIR / "job_descriptions.csv").fillna("")
resume_df = pd.read_csv(DATA_DIR / "resumes.csv").fillna("")
labels_df = pd.read_csv(DATA_DIR / "resume_job_labels.csv")

print("Job descriptions:", job_df.shape)
print("Resumes:", resume_df.shape)
print("Human labels:", labels_df.shape)
display(job_df.head(3))
display(resume_df[["resume_id", "total_experience_years", "source_format", "layout_difficulty"]].head())
display(labels_df.head())

Job descriptions: (10, 11)
Resumes: (185, 15)
Human labels: (555, 7)


,job_id,job_title,department_id,job_level,min_experience_years,required_skills,nice_to_have_skills,education_preference,work_location,openings,description_text
0,J01,Software Engineer,D01,2,2,Python; SQL; REST APIs; Git; Data Structures,Docker; AWS; Django; FastAPI; PostgreSQL; Unit...,B.E./B.Tech in Computer Science or related,Bengaluru / Chennai / Hyderabad (hybrid),4,Zenith Nova Technologies is hiring a Software ...
1,J02,Senior Software Engineer,D01,3,6,Python; System Design; REST APIs; PostgreSQL; ...,Kubernetes; AWS; Microservices; CI/CD; Mentoring,B.E./B.Tech or MCA,Bengaluru / Chennai / Hyderabad (hybrid),3,Zenith Nova Technologies is hiring a Senior So...
2,J03,Data Analyst,D03,2,2,SQL; Excel; Python; Data Visualization; Statis...,Power BI; Tableau; pandas; A/B Testing; Apache...,Bachelor's in a quantitative field,Bengaluru / Chennai / Hyderabad (hybrid),4,Zenith Nova Technologies is hiring a Data Anal...


,resume_id,total_experience_years,source_format,layout_difficulty
0,RES-0001,4.8,DOCX,Missing sections
1,RES-0002,3.7,PDF,Clean
2,RES-0003,5.7,PDF,Clean
3,RES-0004,5.1,DOCX,Clean
4,RES-0005,1.4,PDF,Clean


,resume_id,job_id,human_suitability_score,shortlist,required_skill_coverage,nice_skill_coverage,reviewer_note
0,RES-0001,J04,4,Y,1.0,0.6,Strong match
1,RES-0001,J09,2,N,0.0,0.0,Weak match
2,RES-0001,J03,2,N,0.4,0.0,Weak match
3,RES-0002,J07,3,N,0.6,0.5,Partial match
4,RES-0002,J05,2,N,0.0,0.0,Weak match


In [3]:
required_job_cols = {"job_id", "job_title", "min_experience_years", "required_skills", "nice_to_have_skills", "description_text"}
required_resume_cols = {"resume_id", "total_experience_years", "skills", "resume_text"}
required_label_cols = {"resume_id", "job_id", "human_suitability_score", "shortlist", "required_skill_coverage", "nice_skill_coverage", "reviewer_note"}
assert required_job_cols.issubset(job_df.columns)
assert required_resume_cols.issubset(resume_df.columns)
assert required_label_cols.issubset(labels_df.columns)
assert job_df.job_id.is_unique and resume_df.resume_id.is_unique
assert not labels_df.duplicated(["resume_id", "job_id"]).any()
assert set(labels_df.resume_id).issubset(set(resume_df.resume_id))
assert set(labels_df.job_id).issubset(set(job_df.job_id))
print("Validation passed: unique IDs, valid foreign keys, no duplicate labeled pairs.")
print("Labels per resume:")
display(labels_df.groupby("resume_id").size().value_counts().sort_index().rename("resume_count"))
print("Suitability score distribution:")
display(labels_df.human_suitability_score.value_counts().sort_index())
print("Shortlist distribution:")
display(labels_df.shortlist.value_counts())

Validation passed: unique IDs, valid foreign keys, no duplicate labeled pairs.
Labels per resume:


3    185
Name: resume_count, dtype: int64

Suitability score distribution:


human_suitability_score
1    106
2    274
3     80
4     72
5     23
Name: count, dtype: int64

Shortlist distribution:


shortlist
N    460
Y     95
Name: count, dtype: int64

## 3. Inspect human-label structure

The 555 labels cover three job pairs per resume, not every possible resume–job pair. The ranking engine will score all 185 × 10 = 1,850 pairs. Evaluation metrics will use only the labeled pairs, and this limitation will be stated in the report.

In [4]:
print("All possible pairs:", len(resume_df) * len(job_df))
print("Labeled pairs:", len(labels_df))
print("Human score vs shortlist:")
display(pd.crosstab(labels_df.human_suitability_score, labels_df.shortlist))
print("Job descriptions and required skills:")
display(job_df[["job_id", "job_title", "min_experience_years", "required_skills", "nice_to_have_skills"]])

All possible pairs: 1850
Labeled pairs: 555
Human score vs shortlist:


shortlist,N,Y
human_suitability_score,,
1,106,0
2,274,0
3,80,0
4,0,72
5,0,23


Job descriptions and required skills:


,job_id,job_title,min_experience_years,required_skills,nice_to_have_skills
0,J01,Software Engineer,2,Python; SQL; REST APIs; Git; Data Structures,Docker; AWS; Django; FastAPI; PostgreSQL; Unit...
1,J02,Senior Software Engineer,6,Python; System Design; REST APIs; PostgreSQL; ...,Kubernetes; AWS; Microservices; CI/CD; Mentoring
2,J03,Data Analyst,2,SQL; Excel; Python; Data Visualization; Statis...,Power BI; Tableau; pandas; A/B Testing; Apache...
3,J04,ML Engineer,3,Python; Machine Learning; scikit-learn; SQL; M...,TensorFlow; PyTorch; MLOps; Docker; NLP
4,J05,Product Manager,5,Product Roadmaps; User Research; Stakeholder M...,Figma; SQL; A/B Testing; Go-to-Market
5,J06,Customer Support Executive,1,Customer Communication; Ticketing Tools; Probl...,Zendesk; CRM; Regional Language Fluency; Escal...
6,J07,HR Executive,2,Recruitment; Employee Onboarding; HR Policies;...,HRIS; Payroll Basics; Labour Law Basics; Engag...
7,J08,Accountant,2,Tally; GST; Bookkeeping; MS Excel; Bank Reconc...,TDS; Financial Reporting; SAP; Audit Support
8,J09,Account Executive,3,B2B Sales; CRM; Lead Generation; Negotiation; ...,Salesforce; Cold Outreach; Account Planning; S...
9,J10,Digital Marketing Executive,2,SEO; Content Marketing; Google Analytics; Soci...,Google Ads; HubSpot; Copywriting; A/B Testing


## 4. Clean resume text and build a skill vocabulary

The skill vocabulary comes from the required and nice-to-have skills listed in the job descriptions. We search the cleaned resume text for these phrases. The cleaning step removes candidate identity/contact lines, employer names in structured role headings, and education sections so that names, locations, employer identity and college tier do not affect similarity.

In [5]:
def normalize_text(value):
    value = str(value).lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


company_names = sorted({str(x).strip() for x in resume_df.current_company if str(x).strip()}, key=len, reverse=True)

def clean_resume_text(text):
    lines = str(text).splitlines()
    kept = []
    section = None
    for line_no, line in enumerate(lines):
        value = line.strip()
        if not value:
            continue
        if line_no < 2:  # candidate name and contact/location header
            continue
        if re.fullmatch(r"(SUMMARY|EXPERIENCE|EXPERIENCE NOTES|SKILLS|EDUCATION|CERTIFICATIONS)", value, re.I):
            section = value.upper()
            continue
        if section == "EDUCATION":
            continue
        if "@" in value:
            continue
        value = re.sub(r"\bTier\s*[123]\s*institution\b", "", value, flags=re.I)
        for company in company_names:
            value = re.sub(re.escape(company), "", value, flags=re.I)
        if section == "EXPERIENCE" and not value.startswith("-") and "(" in value and ")" in value:
            match = re.match(r"^(.+?)[,|]\s*.+?\s*(\([0-9.]+\s*yrs?\))$", value, re.I)
            if match:
                value = f"{match.group(1)} {match.group(2)}"
        value = re.sub(r"\s+", " ", value).strip()
        if value:
            kept.append(value)
    return " ".join(kept)

job_skill_terms = sorted({
    skill.strip()
    for col in ["required_skills", "nice_to_have_skills"]
    for value in job_df[col]
    for skill in str(value).split(";")
    if skill.strip()
}, key=len, reverse=True)

resume_df["clean_resume_text"] = resume_df.resume_text.map(clean_resume_text)
print("Unique job-skill terms:", len(job_skill_terms))
print("Example cleaned resume text:")
print(resume_df.loc[0, "clean_resume_text"][:700])

Unique job-skill terms: 80
Example cleaned resume text:
Experienced ml engineer with 4.8 years of experience. Strengths include MLOps, Communication, TensorFlow. ML Engineer (2.2 yrs) - Trained models that improved accuracy by 28% - Built feature pipelines for 44 models Data Analyst (2.6 yrs) - Built feature pipelines for 29 models MLOps, Communication, TensorFlow, scikit-learn, PyTorch, Machine Learning, Python, Model Deployment, System Design, SQL, MS Office


In [6]:
extracted_skill_sets = []
for text in resume_df.clean_resume_text:
    normalized = " " + normalize_text(text) + " "
    extracted_skill_sets.append({
        normalize_text(skill) for skill in job_skill_terms
        if (" " + normalize_text(skill) + " ") in normalized
    })
resume_df["extracted_skills"] = extracted_skill_sets
print("Example extracted skills:")
print(resume_df.loc[0, "resume_id"], sorted(resume_df.loc[0, "extracted_skills"]))

Example extracted skills:
RES-0001 ['communication', 'machine learning', 'mlops', 'model deployment', 'python', 'pytorch', 'scikit learn', 'sql', 'system design', 'tensorflow']


## 5. Audit skill extraction quality

The separate `skills` column is used here only as a reference to audit extraction. It is not used in the ranking score. This audit helps identify layout-related extraction weaknesses.

In [7]:
extraction_rows = []
for _, row in resume_df.iterrows():
    detected = row.extracted_skills
    listed = {normalize_text(x) for x in str(row.skills).split(";") if x.strip()}
    extraction_rows.append({
        "resume_id": row.resume_id,
        "layout_difficulty": row.layout_difficulty,
        "detected_skill_count": len(detected),
        "listed_skill_count": len(listed),
        "matched_listed_skills": len(detected & listed),
        "missed_listed_skills": len(listed - detected),
        "extra_detected_skills": len(detected - listed),
    })
extraction_audit = pd.DataFrame(extraction_rows)
TP = int(extraction_audit.matched_listed_skills.sum())
FP = int(extraction_audit.extra_detected_skills.sum())
FN = int(extraction_audit.missed_listed_skills.sum())
extract_precision = TP / (TP + FP) if TP + FP else 0
extract_recall = TP / (TP + FN) if TP + FN else 0
extract_f1 = 2 * extract_precision * extract_recall / (extract_precision + extract_recall) if extract_precision + extract_recall else 0
print({"precision": round(extract_precision, 3), "recall": round(extract_recall, 3), "f1": round(extract_f1, 3)})
layout_audit = extraction_audit.groupby("layout_difficulty", as_index=False).agg(
    detected_skills=("detected_skill_count", "sum"),
    listed_skills=("listed_skill_count", "sum"),
    matched_listed_skills=("matched_listed_skills", "sum"),
    missed_listed_skills=("missed_listed_skills", "sum"),
)
layout_audit["listed_skill_recall"] = layout_audit.matched_listed_skills / layout_audit.listed_skills.replace(0, np.nan)
display(layout_audit.sort_values("listed_skill_recall"))

{'precision': 0.937, 'recall': 0.781, 'f1': 0.852}


,layout_difficulty,detected_skills,listed_skills,matched_listed_skills,missed_listed_skills,listed_skill_recall
2,Missing sections,116,180,112,68,0.622222
0,All-caps,130,150,118,32,0.786667
3,Tables,177,209,166,43,0.794258
1,Clean,698,818,657,161,0.803178
4,Two-column,198,225,183,42,0.813333


## 6. TF-IDF similarity

TF-IDF turns text into weighted numeric features. Cosine similarity compares each cleaned resume with each job description. The vectorizer uses unigrams and bigrams; it does not use the human labels.

In [8]:
job_texts = (
    job_df.job_title.astype(str) + " " +
    job_df.required_skills.astype(str) + " " +
    job_df.nice_to_have_skills.astype(str) + " " +
    job_df.description_text.astype(str)
).tolist()
corpus = resume_df.clean_resume_text.tolist() + job_texts
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
text_matrix = vectorizer.fit_transform(corpus)
resume_matrix = text_matrix[:len(resume_df)]
job_matrix = text_matrix[len(resume_df):]
cosine_matrix = cosine_similarity(job_matrix, resume_matrix)
print("TF-IDF matrix:", text_matrix.shape)
print("Cosine matrix (jobs × resumes):", cosine_matrix.shape)

TF-IDF matrix: (195, 3062)
Cosine matrix (jobs × resumes): (10, 185)


## 7. Build a transparent ranking score

The weights are fixed before evaluating against human labels:

- Required-skill coverage: 50%
- Nice-to-have skill coverage: 15%
- TF-IDF cosine similarity: 20%
- Experience fit: 15%

Experience fit is `min(candidate_years / minimum_required_years, 1)`. Extra years above the minimum do not add extra points. Education and college tier are not scored.

In [9]:
score_rows = []
for job_index, job in job_df.iterrows():
    required = {normalize_text(x) for x in str(job.required_skills).split(";") if x.strip()}
    nice = {normalize_text(x) for x in str(job.nice_to_have_skills).split(";") if x.strip()}
    for resume_index, resume in resume_df.iterrows():
        found = resume.extracted_skills
        required_coverage = len(required & found) / len(required) if required else 0.0
        nice_coverage = len(nice & found) / len(nice) if nice else 0.0
        experience_fit = min(float(resume.total_experience_years) / max(float(job.min_experience_years), 1.0), 1.0)
        tfidf_similarity = float(cosine_matrix[job_index, resume_index])
        match_score = (
            0.50 * required_coverage +
            0.15 * nice_coverage +
            0.20 * tfidf_similarity +
            0.15 * experience_fit
        )
        score_rows.append({
            "job_id": job.job_id,
            "job_title": job.job_title,
            "resume_id": resume.resume_id,
            "match_score": match_score,
            "required_skill_coverage": required_coverage,
            "nice_skill_coverage": nice_coverage,
            "tfidf_similarity": tfidf_similarity,
            "experience_fit": experience_fit,
            "experience_years": float(resume.total_experience_years),
        })
rankings = pd.DataFrame(score_rows)
rankings["rank"] = rankings.groupby("job_id").match_score.rank(method="first", ascending=False).astype(int)
rankings = rankings.sort_values(["job_id", "rank"]).reset_index(drop=True)
print("Scored pairs:", len(rankings))
display(rankings.groupby("job_id").head(5))

Scored pairs: 1850


,job_id,job_title,resume_id,match_score,required_skill_coverage,nice_skill_coverage,tfidf_similarity,experience_fit,experience_years,rank
0,J01,Software Engineer,RES-0138,0.815603,1.0,0.833333,0.203014,1.000000,6.4,1
1,J01,Software Engineer,RES-0123,0.790004,1.0,0.666667,0.200018,1.000000,6.2,2
2,J01,Software Engineer,RES-0161,0.790004,1.0,0.666667,0.200018,1.000000,6.2,3
3,J01,Software Engineer,RES-0151,0.719513,0.8,0.833333,0.222565,1.000000,3.2,4
4,J01,Software Engineer,RES-0112,0.674677,0.8,0.666667,0.123385,1.000000,3.5,5
185,J02,Senior Software Engineer,RES-0102,0.685653,0.8,0.600000,0.228265,1.000000,7.9,1
186,J02,Senior Software Engineer,RES-0123,0.605987,0.8,0.200000,0.129934,1.000000,6.2,2
187,J02,Senior Software Engineer,RES-0161,0.605987,0.8,0.200000,0.129934,1.000000,6.2,3
188,J02,Senior Software Engineer,RES-0042,0.598578,0.6,0.800000,0.242890,0.866667,5.2,4
189,J02,Senior Software Engineer,RES-0129,0.595879,0.6,0.800000,0.129397,1.000000,6.1,5


## 8. Evaluate against human judgments

Labels are joined only after the ranking scores are calculated. Metrics are computed within each job using its labeled candidate subset. Precision@5 measures the share of the model's top five labeled candidates that humans shortlisted; Recall@5 measures the share of all human-shortlisted labeled candidates found in that top five. NDCG@5 uses the 1–5 suitability score as graded relevance.

In [10]:
labeled_rankings = labels_df.merge(rankings, on=["resume_id", "job_id"], how="left", validate="one_to_one")

def ndcg_at_k(relevance, k=5):
    values = np.asarray(relevance, dtype=float)
    gains = np.power(2.0, values) - 1.0
    discounts = np.log2(np.arange(2, min(len(values), k) + 2))
    dcg = float(np.sum(gains[:k] / discounts))
    ideal = np.sort(gains)[::-1][:k]
    ideal_discounts = np.log2(np.arange(2, len(ideal) + 2))
    idcg = float(np.sum(ideal / ideal_discounts))
    return dcg / idcg if idcg else 0.0

metric_rows = []
for job_id, group in labeled_rankings.groupby("job_id"):
    ordered = group.sort_values("match_score", ascending=False)
    top5 = ordered.head(5)
    total_shortlisted = int((ordered.shortlist == "Y").sum())
    metric_rows.append({
        "job_id": job_id,
        "job_title": ordered.job_title.iloc[0],
        "labeled_pairs": len(ordered),
        "shortlisted_pairs": total_shortlisted,
        "precision_at_5": (top5.shortlist == "Y").mean(),
        "recall_at_5": (top5.shortlist == "Y").sum() / total_shortlisted if total_shortlisted else 0.0,
        "ndcg_at_5": ndcg_at_k(ordered.human_suitability_score.to_numpy(), 5),
        "spearman_vs_human_score": spearmanr(ordered.match_score, ordered.human_suitability_score).statistic,
    })
metrics = pd.DataFrame(metric_rows)
macro_metrics = metrics[["precision_at_5", "recall_at_5", "ndcg_at_5", "spearman_vs_human_score"]].mean()
display(metrics)
print("Macro-average metrics:")
display(macro_metrics.to_frame("value"))

,job_id,job_title,labeled_pairs,shortlisted_pairs,precision_at_5,recall_at_5,ndcg_at_5,spearman_vs_human_score
0,J01,Software Engineer,64,7,1.0,0.714286,1.000000,0.834912
1,J02,Senior Software Engineer,47,3,0.4,0.666667,0.893862,0.865171
2,J03,Data Analyst,60,12,1.0,0.416667,1.000000,0.830773
3,J04,ML Engineer,62,16,1.0,0.312500,0.825251,0.934226
4,J05,Product Manager,57,9,1.0,0.555556,1.000000,0.805279
5,J06,Customer Support Executive,49,6,1.0,0.833333,1.000000,0.616434
6,J07,HR Executive,49,8,1.0,0.625000,1.000000,0.821834
7,J08,Accountant,55,9,1.0,0.555556,1.000000,0.833046
8,J09,Account Executive,57,14,1.0,0.357143,0.889555,0.906514
9,J10,Digital Marketing Executive,55,11,1.0,0.454545,0.991771,0.829803


Macro-average metrics:


,value
precision_at_5,0.940000
recall_at_5,0.549125
ndcg_at_5,0.960044
spearman_vs_human_score,0.827799


## 9. Inspect five misranked examples

The following cases compare the model's rank across all 185 resumes for a job with the human label. A high model rank is not proof of suitability; these disagreements show where simple text matching and weighted features can fail.

In [11]:
misrank_specs = [
    ("J02", "RES-0129", "Over-ranking: the model places this candidate 5th (score 0.596) because it detects 80% of nice-to-have skills and full experience fit. However, required-skill coverage is only 60%; System Design and REST APIs are not detected. The reviewer marked a partial match (score 3/5, not shortlisted). The weighted score allows preferred skills and experience to compensate too much for missing core requirements."),
    ("J03", "RES-0101", "Over-ranking: the model places this candidate 11th (score 0.607) with 80% required-skill coverage and full experience fit, but Statistics is missing and the human reviewer marked a partial match (3/5, not shortlisted). Exact phrase coverage does not judge the depth or evidence behind a skill, and the experience component helps offset the missing requirement."),
    ("J08", "RES-0087", "Over-ranking relative to the reviewer: the model detects all five required skill phrases and ranks the candidate 8th (score 0.647), but the candidate has only 1.2 years against the 2-year minimum (experience fit 0.60). The reviewer assessed required-skill coverage as 60% and marked a partial match (3/5, not shortlisted). This exposes a difference between literal phrase detection and human assessment of demonstrated competence."),
    ("J04", "RES-0130", "Under-ranking: the candidate is a human-rated strong match (4/5, shortlisted; reviewer required-skill coverage 100%) but ranks 25th. The resume is marked as having missing sections and has no explicit SKILLS section in resume_text. The text-only extractor detects only 40% of required skills, despite 6.9 years of experience. The structured skills column contains additional relevant terms but is intentionally not used in the score."),
    ("J09", "RES-0012", "Under-ranking: the candidate is a human-rated strong match (5/5, shortlisted; reviewer required-skill coverage 80% and nice-to-have coverage 75%) but ranks 19th. The resume has missing sections and no explicit SKILLS section in resume_text. Text extraction detects only 40% of required skills and 25% of preferred skills. Relevant terms exist in the separate structured skills column, which the text-only ranking deliberately does not use."),
]
misrank_rows = []
for job_id, resume_id, explanation in misrank_specs:
    scored = rankings[(rankings.job_id == job_id) & (rankings.resume_id == resume_id)].iloc[0]
    human = labels_df[(labels_df.job_id == job_id) & (labels_df.resume_id == resume_id)].iloc[0]
    misrank_rows.append({
        "job_id": job_id, "job_title": scored.job_title, "resume_id": resume_id,
        "model_rank_all_185": int(scored["rank"]), "match_score": scored.match_score,
        "required_skill_coverage": scored.required_skill_coverage,
        "nice_skill_coverage": scored.nice_skill_coverage,
        "tfidf_similarity": scored.tfidf_similarity,
        "experience_fit": scored.experience_fit,
        "human_suitability_score": human.human_suitability_score,
        "human_shortlist": human.shortlist, "reviewer_note": human.reviewer_note,
        "analysis": explanation,
    })
misrankings = pd.DataFrame(misrank_rows)
display(misrankings)


,job_id,job_title,resume_id,model_rank_all_185,match_score,required_skill_coverage,nice_skill_coverage,tfidf_similarity,experience_fit,human_suitability_score,human_shortlist,reviewer_note,analysis
0,J02,Senior Software Engineer,RES-0129,5,0.595879,0.6,0.80,0.129397,1.0,3,N,Partial match,Over-ranking: the model places this candidate ...
1,J03,Data Analyst,RES-0101,11,0.607065,0.8,0.20,0.135323,1.0,3,N,Partial match,Over-ranking: the model places this candidate ...
2,J08,Accountant,RES-0087,8,0.647489,1.0,0.25,0.099944,0.6,3,N,Partial match,Over-ranking relative to the reviewer: the mod...
3,J04,ML Engineer,RES-0130,25,0.375973,0.4,0.00,0.129867,1.0,4,Y,Strong match,Under-ranking: the candidate is a human-rated ...
4,J09,Account Executive,RES-0012,19,0.402983,0.4,0.25,0.077417,1.0,5,Y,Strong match,Under-ranking: the candidate is a human-rated ...


## 10. Representative manual review: five resumes and three job descriptions

The task specifically asks for a close read of five resumes against three job descriptions. This focused review uses **J04 (ML Engineer), J08 (Accountant), and J09 (Account Executive)** and five actual resumes. The examples include human-rated strong matches and cases where layout/text extraction causes a ranking disagreement.

A good match should demonstrate the job's required skills in the resume, include relevant preferred skills where possible, and meet the minimum experience requirement. Preferred skills should not compensate completely for missing essential requirements. Candidate identity, age, gender, location, employer identity, and college tier are not match criteria.

In [12]:
review_job_ids = ["J04", "J08", "J09"]
review_resume_ids = ["RES-0031", "RES-0130", "RES-0010", "RES-0007", "RES-0012"]

review_jobs = job_df[job_df.job_id.isin(review_job_ids)][[
    "job_id", "job_title", "min_experience_years", "required_skills", "nice_to_have_skills"
]].copy()
print("Three job descriptions selected for close review:")
display(review_jobs)

review_resumes = resume_df[resume_df.resume_id.isin(review_resume_ids)][[
    "resume_id", "total_experience_years", "source_format", "layout_difficulty", "resume_text"
]].copy()
review_rows = []
for _, resume in review_resumes.iterrows():
    # Pair each selected resume with the relevant reviewed job based on its labeled match.
    candidate_labels = labels_df[
        (labels_df.resume_id == resume.resume_id) & (labels_df.job_id.isin(review_job_ids))
    ].sort_values("human_suitability_score", ascending=False)
    if candidate_labels.empty:
        continue
    human = candidate_labels.iloc[0]
    scored = rankings[(rankings.resume_id == resume.resume_id) & (rankings.job_id == human.job_id)].iloc[0]
    review_rows.append({
        "resume_id": resume.resume_id,
        "job_id": human.job_id,
        "job_title": scored.job_title,
        "experience_years": resume.total_experience_years,
        "source_format": resume.source_format,
        "layout_difficulty": resume.layout_difficulty,
        "model_rank_all_185": int(scored["rank"]),
        "model_score": round(float(scored.match_score), 4),
        "model_required_skill_coverage": round(float(scored.required_skill_coverage), 2),
        "human_suitability_score": int(human.human_suitability_score),
        "human_shortlist": human.shortlist,
        "reviewer_note": human.reviewer_note,
    })
representative_review = pd.DataFrame(review_rows).sort_values(["job_id", "resume_id"])
assert representative_review.resume_id.nunique() == 5
assert representative_review.job_id.nunique() == 3
print("Five-resume / three-job representative review:")
display(representative_review)
print("Good-match criteria: required skills evidenced in resume text; relevant preferred skills; experience meeting the stated minimum; no identity or demographic attributes.")


Three job descriptions selected for close review:


,job_id,job_title,min_experience_years,required_skills,nice_to_have_skills
3,J04,ML Engineer,3,Python; Machine Learning; scikit-learn; SQL; M...,TensorFlow; PyTorch; MLOps; Docker; NLP
7,J08,Accountant,2,Tally; GST; Bookkeeping; MS Excel; Bank Reconc...,TDS; Financial Reporting; SAP; Audit Support
8,J09,Account Executive,3,B2B Sales; CRM; Lead Generation; Negotiation; ...,Salesforce; Cold Outreach; Account Planning; S...


Five-resume / three-job representative review:


,resume_id,job_id,job_title,experience_years,source_format,layout_difficulty,model_rank_all_185,model_score,model_required_skill_coverage,human_suitability_score,human_shortlist,reviewer_note
3,RES-0031,J04,ML Engineer,4.3,PDF,Clean,2,0.7795,1.0,5,Y,Strong match
4,RES-0130,J04,ML Engineer,6.9,DOCX,Missing sections,25,0.3760,0.4,4,Y,Strong match
1,RES-0010,J08,Accountant,3.4,DOCX,Clean,7,0.6644,0.8,4,Y,Strong match
0,RES-0007,J09,Account Executive,2.8,PDF,Two-column,4,0.7894,1.0,5,Y,Strong match
2,RES-0012,J09,Account Executive,3.7,Plain text,Missing sections,19,0.4030,0.4,5,Y,Strong match


Good-match criteria: required skills evidenced in resume text; relevant preferred skills; experience meeting the stated minimum; no identity or demographic attributes.


### What the five-resume review shows

- **RES-0031 / ML Engineer:** selected as a human-rated strong match; use it as a positive comparison case for the ML Engineer requirements.
- **RES-0130 / ML Engineer:** human-rated strong match but ranked low because its text has missing sections; text-only skill extraction under-detects relevant capabilities.
- **RES-0010 / Accountant:** human-rated strong match and provides a positive comparison for accounting requirements.
- **RES-0007 / Account Executive:** human-rated strong match and provides a positive comparison for sales requirements.
- **RES-0012 / Account Executive:** human-rated strong match but ranked lower because relevant skills in the structured skills field are not fully represented in the resume text used by the scorer.

This is a small, illustrative manual review, not a separate statistically representative sample. The complete ranking and metrics continue to use the full available data.

## 11. Fairness and feature-use checklist

The score does not use candidate names, email addresses, city, employer identity, education/college tier, `target_job_family`, reviewer notes, human suitability scores, or shortlist labels. No gender or age fields are present in the supplied resume schema. Years of experience is capped at the job minimum so that more years alone do not keep increasing the score. Ranking on experience alone could disadvantage early-career candidates, career changers, and people returning to work; experience therefore contributes only 15%. Layout difficulty is inspected as a parsing-quality diagnostic, not as a candidate ranking feature. The extractor recovered only about 62.2% of listed skills for resumes marked as missing sections, versus roughly 79–81% for the other layout groups, highlighting a format-related false-negative risk.

In [13]:
excluded_columns = ["candidate_name", "email", "city", "current_company", "highest_education", "college_tier", "target_job_family"]
print("Excluded from score:", excluded_columns)
print("Human label columns used only for evaluation:", ["human_suitability_score", "shortlist", "reviewer_note"])
print("Layout extraction diagnostic:")
display(layout_audit)

Excluded from score: ['candidate_name', 'email', 'city', 'current_company', 'highest_education', 'college_tier', 'target_job_family']
Human label columns used only for evaluation: ['human_suitability_score', 'shortlist', 'reviewer_note']
Layout extraction diagnostic:


,layout_difficulty,detected_skills,listed_skills,matched_listed_skills,missed_listed_skills,listed_skill_recall
0,All-caps,130,150,118,32,0.786667
1,Clean,698,818,657,161,0.803178
2,Missing sections,116,180,112,68,0.622222
3,Tables,177,209,166,43,0.794258
4,Two-column,198,225,183,42,0.813333


## 12. API endpoint design note (not implemented here)

Suggested endpoint: `POST /v1/resume-ranking/rank`.

**Input:** `job_id` and a list of candidate `resume_id` values (a future upload workflow could accept parsed resume text).

**Output:** ordered candidate IDs, rank, total match score, and component scores for required skills, nice-to-have skills, TF-IDF similarity, and experience fit. Do not expose names or contact details in ranking responses. This notebook does not modify the existing FastAPI `main.py`.

## 13. Save outputs

In [14]:
rankings.to_csv(REPORT_DIR / "task11_resume_rankings.csv", index=False)
metrics.to_csv(REPORT_DIR / "task11_ranking_quality_by_job.csv", index=False)
misrankings.to_csv(REPORT_DIR / "task11_five_misranked_resumes.csv", index=False)
extraction_audit.to_csv(REPORT_DIR / "task11_skill_extraction_audit.csv", index=False)
layout_audit.to_csv(REPORT_DIR / "task11_skill_extraction_by_layout.csv", index=False)
print("Saved evidence to:", REPORT_DIR)
representative_review.to_csv(REPORT_DIR / "task11_representative_review.csv", index=False)
print("Saved representative review:", REPORT_DIR / "task11_representative_review.csv")

Saved evidence to: /workspace/phase-3/reports
Saved representative review: /workspace/phase-3/reports/task11_representative_review.csv


## 14. Conclusion

This is a first explainable ranking baseline, not an automated hiring decision system. The evaluation is limited to 555 human-labeled pairs, and dictionary-based skill extraction can miss synonyms or content hidden by unusual layouts. Human review remains necessary, especially for low-confidence or misranked candidates.